# Challenge 04 -- Deploy a Hosted Weather Agent on Foundry

In this challenge, you will review and deploy a **hosted Weather Agent** on Azure AI
Foundry Agent Service. Unlike the prompt-based agents from earlier challenges, this
agent is a **containerized Python application** -- you build it into a Docker image,
deploy it to Foundry, and invoke it through a hosted endpoint.

## Learning Objectives
- Understand the difference between prompt-based and hosted agents
- Review a hosted agent built with the Agent Framework SDK and a custom tool
- Build a container image on Azure Container Registry (no local Docker needed)
- Deploy to Foundry Agent Service via the Python SDK
- Invoke the hosted agent endpoint and verify it works

## How This Notebook Works
- **Pre-filled cells**: Setup, authentication, code review, and building -- just run them.
- **Fill-in-the-blank cells** (marked FILL IN): Deployment and invocation -- you complete the `___` placeholders.
- Each `___` has a `# TODO:` comment explaining what goes there.

---
## Setup -- Install Dependencies

Run this cell once to install packages needed for this notebook.

In [ ]:
%pip install azure-ai-projects azure-identity python-dotenv pyyaml --quiet

---
## Setup -- Configure Your Project

Set your Foundry project endpoint, model deployment name, and ACR name below.
These values come from your Challenge 00 infrastructure deployment.

**Update the three values below to match your environment, then run the cell.**

In [ ]:
import os

# --- Update these to match your environment ---
PROJECT_ENDPOINT = "<your-foundry-project-endpoint>"
MODEL_DEPLOYMENT_NAME = "gpt-4-1"
ACR_NAME = "<your-acr-name>"

# Validate
assert PROJECT_ENDPOINT != "<your-foundry-project-endpoint>", "Update PROJECT_ENDPOINT above"
assert ACR_NAME != "<your-acr-name>", "Update ACR_NAME above"
assert "/api/projects/" in PROJECT_ENDPOINT, (
    "PROJECT_ENDPOINT must be a Foundry project endpoint, e.g. "
    "https://<resource>.services.ai.azure.com/api/projects/<project-name>"
)

print(f"Endpoint: {PROJECT_ENDPOINT}")
print(f"Model:    {MODEL_DEPLOYMENT_NAME}")
print(f"ACR:      {ACR_NAME}")

---
## Setup -- Authenticate

This uses `ChainedTokenCredential` -- it tries Azure CLI first, then falls back to
interactive browser login. Same pattern as Challenges 02 and 03.

**Prerequisite:** Run `az login` in your terminal before running this cell.

In [ ]:
from azure.identity import AzureCliCredential, InteractiveBrowserCredential, ChainedTokenCredential

TENANT_ID = os.getenv("AZURE_TENANT_ID")

credential = ChainedTokenCredential(
    AzureCliCredential(tenant_id=TENANT_ID) if TENANT_ID else AzureCliCredential(),
    InteractiveBrowserCredential(tenant_id=TENANT_ID) if TENANT_ID else InteractiveBrowserCredential(),
)

# Quick test -- get a token to verify credentials work
credential.get_token("https://management.azure.com/.default")
print("Connected to Azure")

---
## Review the Agent Code

Before deploying, review the pre-provided hosted agent code. The `weather-agent/`
directory contains all the files needed to build and deploy the agent.

The key file is `main.py` -- it defines a weather tool, creates an Agent with
FoundryChatClient, and starts a ResponsesHostServer on port 8088.

**Read through the code below and make sure you understand each section before
moving on to deployment.**

In [ ]:
# Display main.py with line numbers
print("=" * 60)
print("weather-agent/main.py")
print("=" * 60)
with open("weather-agent/main.py") as f:
    for i, line in enumerate(f, 1):
        print(f"{i:3d}  {line}", end="")

print("\n\nKey components:")
print("  - @tool decorator: defines the get_weather function the agent can call")
print("  - FoundryChatClient: connects to your model deployment via DefaultAzureCredential")
print("  - Agent: combines the client, instructions, and tools")
print("  - ResponsesHostServer: exposes the agent as an HTTP endpoint (Responses protocol)")

In [ ]:
import yaml

# Display agent.yaml
print("=" * 60)
print("weather-agent/agent.yaml")
print("=" * 60)
with open("weather-agent/agent.yaml") as f:
    content = f.read()
    print(content)

data = yaml.safe_load(content)
print(f"  kind:      {data.get('kind')} (hosted = runs in your container)")
print(f"  protocol:  {data['protocols'][0]['protocol']} (HTTP API format)")
print(f"  resources: {data['resources']['cpu']} CPU, {data['resources']['memory']} memory")

# Display Dockerfile
print("\n" + "=" * 60)
print("weather-agent/Dockerfile")
print("=" * 60)
with open("weather-agent/Dockerfile") as f:
    print(f.read())

---
## Part 1: Build the Container Image

This uses `az acr build` to build the Docker image remotely on Azure Container Registry.
No local Docker installation needed -- ACR does the build in the cloud.

This typically takes about 2 minutes. Just run the cell and wait.

In [ ]:
import json as _json
import subprocess

print("Building container image on ACR (this takes about 2 minutes)...")

result = subprocess.run(
    f"az acr build --registry {ACR_NAME} --image weather-agent:v1 "
    f"--platform linux/amd64 weather-agent/ --no-logs",
    shell=True, capture_output=True, text=True,
)

if result.returncode == 0:
    build_info = _json.loads(result.stdout)
    status = build_info.get("status", "unknown")
    images = build_info.get("outputImages", [])
    print(f"Build status: {status}")
    if images:
        print(f"Image: {images[0]['registry']}/{images[0]['repository']}:{images[0]['tag']}")
    if status == "Succeeded":
        print("[OK] Container image built successfully")
    else:
        print("[WARN] Build finished but status is: " + status)
else:
    print("[FAIL] Build failed:")
    print(result.stderr[:500])

---
## FILL IN -- Part 2: Deploy the Agent to Foundry

Use `AIProjectClient` to create an agent version that points to the container image
you just built. The SDK sends the agent definition to Foundry, which provisions a
container and makes the agent available as a hosted endpoint.

`HostedAgentDefinition` specifies: the container image, the protocol (Responses),
CPU/memory resources, and environment variables to inject at runtime.

**Fill in the 4 blanks below.**

In [ ]:
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import HostedAgentDefinition, ProtocolVersionRecord, AgentProtocol

project = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=credential,
    allow_preview=True,
)

# Check if agent already exists and is active (avoid creating duplicate versions)
try:
    existing = project.agents.get_version(agent_name="weather-agent", agent_version=1)
    if existing["status"] == "active":
        print("Agent weather-agent v1 is already active -- skipping creation")
        agent_version = existing
    else:
        raise Exception("exists but not active, recreate")
except Exception:
    image_url = f"{ACR_NAME}.azurecr.io/weather-agent:v1"

    agent_version = project.agents.create_version(
        agent_name="weather-agent",
        definition=___(  # TODO: What class defines a hosted agent? (Hint: imported above)
            container_protocol_versions=[
                ProtocolVersionRecord(protocol=AgentProtocol.___, version="1.0.0")
                # TODO: What protocol does the agent use? RESPONSES or INVOCATIONS?
            ],
            cpu="0.5",
            memory="1Gi",
            image=___,  # TODO: What variable holds the full ACR image URL?
            environment_variables={
                "AZURE_AI_MODEL_DEPLOYMENT_NAME": ___,
                # TODO: What variable holds the model deployment name?
            },
        ),
    )
    print(f"Agent created: {agent_version.name}, version: {agent_version.version}")

### Wait for Deployment

This cell polls the agent status until it becomes active. Typically takes under a minute.

In [ ]:
import time

version_num = agent_version["version"] if isinstance(agent_version, dict) else agent_version.version

while True:
    info = project.agents.get_version(
        agent_name="weather-agent",
        agent_version=version_num,
    )
    status = info["status"]
    print(f"Status: {status}")
    if status == "active":
        print("[OK] Agent is deployed and ready")
        break
    elif status == "failed":
        print(f"[FAIL] Provisioning failed: {info.get('error', 'unknown')}")
        break
    time.sleep(10)

---
## FILL IN -- Part 3: Invoke the Deployed Agent

Now send a request to the hosted agent running on Foundry. Use
`project.get_openai_client()` bound to your agent name -- it returns an OpenAI-compatible
client that routes requests to your hosted container.

**Fill in the 3 blanks below.**

In [ ]:
# TODO: Get an OpenAI client bound to your hosted agent
openai_client = project.get_openai_client(agent_name="___")
# TODO: What is your agent's name?

response = openai_client.responses.create(
    input="___",  # TODO: Ask the agent about the weather in a city
)

print("Deployed agent says:")
print(response.output_text)

### FILL IN -- Second Query

Send another query to test that the agent handles different cities.

**Fill in the 2 blanks below.**

In [ ]:
response2 = openai_client.responses.create(
    input="___",  # TODO: Ask about the weather in a different city
)

print("Deployed agent says:")
print(___.___)  # TODO: Print the response text (Hint: same as above)

---
## Verify in the Foundry Portal

Open the [Microsoft Foundry portal](https://ai.azure.com) and confirm:
1. Navigate to your project and open the **Agents** section
2. Your Weather Agent appears in the agents list
3. The status shows **Active**

In [ ]:
print("=" * 55)
print("CHALLENGE 04 -- COMPLETION CHECKLIST")
print("=" * 55)
print()
print("[  ]  Reviewed agent code: main.py, agent.yaml, Dockerfile")
print("[  ]  Container image built on ACR")
print("[  ]  Agent deployed via Python SDK (create_version)")
print("[  ]  Agent status is Active")
print("[  ]  Invoked deployed agent -- got weather responses")
print()
print("Show your deployed agent and responses to your coach!")

---
## Key Takeaways

| Concept | Prompt-Based Agents (Ch 01-03) | Hosted Agents (Ch 04) |
|---|---|---|
| **What you write** | System prompt + tool config | Full runtime code (main.py) |
| **Where it runs** | Foundry-managed (serverless) | Your container on Foundry compute |
| **Framework** | Azure AI Agent SDK only | Any framework (Agent Framework, LangGraph, custom) |
| **Protocol** | SDK threads/runs API | HTTP -- Responses or Invocations protocol |
| **Identity** | Shared project identity | Dedicated Entra ID per agent |
| **Deployment** | AgentsClient.create_agent() | SDK create_version() or azd deploy |
| **Best for** | Quick prototyping, simple agents | Production agents, custom logic, multi-framework |

---
## Cleanup

Delete the agent when finished (uncomment to run):
```python
# project.agents.delete(agent_name="weather-agent")
```